# 13 — Placement: machine-bound dims, and collectives with no ops

The ladder so far has been about *what a program means* (L0) and *what it costs*
in operations and memory (L1). This notebook takes the first step of L3 — **where
the computation lives** — and it does so by adding, to the whole library, exactly
**one new field**: a `level` on a `Dim`. No new ops. No collective primitives. No
per-device program.

The thesis, stated as PLACEMENT.md states it: **placement is cost-bearing
metadata, never meaning.** A dim bound to a machine level is an *address*; forget
the bindings and you recover the identical L0 program with the identical
denotation (the **erasure invariant**). Communication is then not something you
*write* — it is something the cost pass *reads off* the ordinary algebra:

> reduce over a bound dim **is** an all-reduce; merge of a bound part **is** an
> all-gather; repeat-then-bind from a replicated source **is** free distribution.

"Collectives are alignment fixes" becomes literal. And once gradients carry their
primal's placement — by the very same restamp that carries charts — **data
parallelism falls out** of the adjoint of a broadcast. That is the finale.


In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
import numpy as np

from pdum.tl import Machine, Tensor, mesh, peak_memory, pointwise, pw, traffic
from pdum.tl.autodiff import grad, numeric_grad
from pdum.tl.ir import Instr, Program, run
from pdum.tl.layout import Dim
from pdum.tl.zoo import megatron_block


def I(var, op, operands=(), **params):  # a bare-hands Program builder, as in the tests
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)

## 1 · The two-artifact principle

There are two artifacts, and the IR knows only one of them. The **machine** is
plain data — a tuple of levels (cluster → … → lane capable), each with a count, a
memory capacity, and a sibling link. The **program** never names hardware; a dim
binds to a *level name* (a string), and the machine is what gives that name
meaning. Swap the machine, retarget the model; the program is untouched
(PLACEMENT.md **D-C**: one mesh level now, a depth-ready schema).

`mesh(2)` is the v1 machine: a single "gpu" mesh, extent 2.


In [2]:
gpu = mesh(2)  # the v1 machine: one mesh level, extent 2
print(gpu)
print("level('gpu') ->", gpu.level("gpu"))

Machine(levels=(Level(name='gpu', count=2, capacity=None, link_bandwidth=None, link_latency=0.0),))
level('gpu') -> Level(name='gpu', count=2, capacity=None, link_bandwidth=None, link_latency=0.0)


## 2 · Binding lives on the `Dim`

PLACEMENT.md **D-A**: a dim is machine-bound via a `level` field — one more
per-dim annotation over the *unchanged* lattice, exactly like charts and labels.
This is not an accident of convenience: **alignment is the collective-inference
engine**, so everything the alignment machinery compares must live in one place.
Binding rides there.

`Tensor.bind(g="gpu")` sets it. In the repr it shows as a `%gpu` suffix — and it
is *pure metadata*: it moves no bytes, so the values are exactly the erasure, and
it survives untouched through views and through compute.


In [3]:
x = T(np.arange(6.0).reshape(2, 3), ("g", "i")).bind(g="gpu")
print("g dim:", repr(x.layout.dim("g")))  # the %gpu suffix *is* the whole binding
print("i dim:", repr(x.layout.dim("i")))
print("g.level =", x.layout.dim("g").level, "   i.level =", x.layout.dim("i").level)

# metadata rides through a view chain and through a pointwise result, untouched...
y = x.slice(i=(0, 2)).repeat("j", 2)
z = pointwise(pw.mul, x, x)
print("rides a view   ->", y.layout.dim("g").level)
print("rides compute  ->", z.layout.dim("g").level)

# ...and the values are exactly the erasure: bind moved no bytes.
np.testing.assert_array_equal(x.to_numpy(), np.arange(6.0).reshape(2, 3))
print("values == erasure: True")

g dim: g[0:2)*24 %gpu
i dim: i[0:3)*8
g.level = gpu    i.level = None
rides a view   -> gpu
rides compute  -> gpu
values == erasure: True


### Surface discipline: bound dims are addresses

A machine-bound dim indexes *devices*, not physics. So it carries **no chart, no
labels** — it is chartless and unlabeled by construction (PLACEMENT.md's surface
discipline; the physics stays on the semantic dims). This is D17 — *diagnosis,
never surgery* — at the point of construction: binding a charted dim, or building
a `Dim` that is both an address and a label, **refuses** rather than silently
dropping one of the two meanings.


In [4]:
# a bound dim is an ADDRESS — the refusal is enforced twice.
try:
    T(np.zeros(3), ("d",)).with_charts(d=(0, 1)).bind(d="gpu")
except ValueError as e:
    print("bind a charted dim  ->", e)
try:
    Dim("d", 8, 0, 3, labels=("a", "b", "c"), level="gpu")
except ValueError as e:
    print("build one directly  ->", e)

bind a charted dim  -> dim d: machine-bound dims are chartless/unlabeled (strip_charts first)
build one directly  -> dim d: machine-bound dims are addresses — chartless and unlabeled


## 3 · The erasure invariant

This is the L3 well-formedness contract, and the whole reason the layer is cheap
to trust (PHILOSOPHY.md, *erasure invariants everywhere*): **forget the bindings
and the program means the same thing.** PLACEMENT.md **D-D** — global view — makes
this exact: one program describes the *global* computation, and the reference
interpreter runs the placed program **unchanged**, computing global values. No
per-device SPMD extraction; `run` needs no changes at all.

The flagship is a Megatron-style tensor-parallel transformer block. Attention
heads and MLP width carry a mesh dim `g`, and `g` is *bound* to `"gpu"`.
`megatron_block(level=None)` builds the **erasure** — the identical program minus
the bindings. Assurance tier 1 (LEVELS.md) is then a single comparison: the placed
run and the erased run must agree **bit-for-bit** (atol=0), and both must match an
independent numpy denotation.


In [5]:
placed = megatron_block()            # heads + mlp width carry g, bound to "gpu"
erased = megatron_block(level=None)  # the SAME program, bindings forgotten

got_p = run(placed.program, placed.inputs)[placed.out].to_numpy(order=placed.order)
got_e = run(erased.program, erased.inputs)[erased.out].to_numpy(order=erased.order)
ref = placed.ref(placed.numpy_inputs())  # an independent numpy denotation

bound = [n for n, t in placed.inputs.items() if any(d.level for d in t.layout.dims)]
print("weights carrying the mesh dim g:", bound)
print("placed vs numpy   max|Δ| =", float(np.max(np.abs(got_p - ref))))
print("placed vs erased  equal at atol=0:", np.array_equal(got_p, got_e))

weights carrying the mesh dim g: ['wq', 'wk', 'wv', 'wo', 'w1', 'b1', 'w2']
placed vs numpy   max|Δ| = 4.440892098500626e-16
placed vs erased  equal at atol=0: True


## 4 · Alignment is the distributed type-checker

If two operands disagree about placement — one sharded on `g`, one replicated —
combining them is a *category error*, and it is exactly the kind of error the
alignment machinery already exists to catch. PLACEMENT.md promotes D17 to L3: a
placement mismatch **refuses**, and the refusal *quotes the fix* — and the fix is
a `bind(...)`, i.e. a (cost-bearing) collective. The misalignment diagnosis's
existing recipes simply *acquire costs* when dims are bound.

Note the asymmetry with the erasure: erasing bindings never changes a value, but
*combining mismatched* bindings is refused, not guessed. Both follow from the same
conviction — the library will not silently invent a collective for you.


In [6]:
xb = T(np.zeros((2, 3)), ("g", "i")).bind(g="gpu")  # sharded on g
yu = T(np.zeros((2, 3)), ("g", "i"))                # replicated
try:
    pointwise(pw.add, xb, yu)
except ValueError as e:
    print(e)  # the quoted fix, bind(g='gpu'), is itself the collective

pointwise(add) requires aligned operands:
  operand 1, dim 'g': machine placement differs (unbound vs gpu)  ->  bind(g='gpu')


## 5 · Collectives with **no ops**

Here is the load-bearing move (PLACEMENT.md **D-B**, PHILOSOPHY.md *collectives
with no ops at all*). The library defines **no collective primitives**. The
`traffic` pass walks an ordinary program, runs the *same* shape inference every
other pass uses, and reads communication off the existing algebra applied to
machine-bound dims:

- **reduce over a bound dim → all-reduce.** The result drops the mesh dim and is
  replicated — which is *exactly* reduce's value semantics. Ring cost:
  $\frac{2(p-1)}{p}\times$ result-local bytes.

Megatron's paper prescribes *exactly two* all-reduces per block — after the
attention output projection and after the MLP down projection. In this IR those
are just the two `reduce`-over-`g` contractions. **Nothing new was defined**; the
traffic pass finds them.


In [7]:
rep = traffic(placed.program, placed.inputs, mesh(2))
for c in rep.collectives:
    print(f"  {c.var:>4}  {c.kind:<11} @{c.level}   {c.bytes} B/device")
print("  per-level totals:", dict(rep.per_level))

# the ring formula by hand, p = 2 so 2(p-1)/p is exactly 1x:
p = 2
result_local = 4 * 6 * 8  # the (t, d) output tensor, per device
print(f"  2(p-1)/p x {result_local} = {(2 * (p - 1) // p) * result_local}  (matches)")

# and the erasure communicates nothing at all — placement is the ONLY difference:
print("  erased collectives:", traffic(erased.program, erased.inputs, mesh(2)).collectives)

     o  all_reduce  @gpu   192 B/device
    m2  all_reduce  @gpu   192 B/device
  per-level totals: {'gpu': 384}
  2(p-1)/p x 192 = 192  (matches)
  erased collectives: ()


### An α-β time estimate

Traffic is *bytes*; turning bytes into *time* is a separate, opinionated cost
model laid over the same report (PHILOSOPHY.md, *meaning first, cost as separate
semantics*). Give a level a `link_bandwidth` and `link_latency` and `rep.time`
sums a latency per collective plus bytes over bandwidth. Topology and overlap are
refinements, not IR (CONCERNS #28).


In [8]:
machine = mesh(2, link_bandwidth=1e9, link_latency=1e-6)  # 1 GB/s, 1 µs/message
print("estimated time:", rep.time(machine), "s")
print("  = 2 msgs x 1e-6 s  +  384 B / 1e9 B/s  =", 2 * 1e-6 + 384 / 1e9)

estimated time: 2.384e-06 s
  = 2 msgs x 1e-6 s  +  384 B / 1e9 B/s  = 2.384e-06


## 6 · Free things (the mesh analogue of "masks are free")

Two moves cost **zero bytes**, as closed forms in the representation:

- **distribute** — repeat a replicated source over a mesh dim, then bind it. Every
  device already holds the value; nothing moves.
- **shard-view** — split an unbound dim and bind one part. That is a *view* of
  data every device already holds; nothing moves.

And one more collective reads straight off the algebra:

- **merge of a bound part → all-gather**, at $\frac{p-1}{p}\times$ merged-global
  bytes. For a $(g{=}2,\, i{=}3)$ tensor of doubles that is $\frac12 \times 48 =
  24$ bytes.


In [9]:
# distribute: repeat a replicated weight over g, then bind -> 0 bytes
dist = Program((
    I("w", "input"),
    I("wr", "repeat", ["w"], name="g", extent=(0, 2)),
    I("wb", "bind", ["wr"], levels={"g": "gpu"}),
))
print("distribute :", traffic(dist, {"w": T(np.zeros(3), ("i",))}, mesh(2)).collectives)

# shard-view: split an unbound dim, bind one part -> a view, 0 bytes
shard = Program((
    I("w", "input"),
    I("ws", "split", ["w"], name="i", parts={"g": 2, "ii": 3}),
    I("wb", "bind", ["ws"], levels={"g": "gpu"}),
))
print("shard-view :", traffic(shard, {"w": T(np.arange(6.0), ("i",))}, mesh(2)).collectives)

# merge of a bound part = all-gather: (p-1)/p x merged bytes = 1/2 x 48 = 24
gather = Program((I("x", "input"), I("m", "merge", ["x"], parts=("g", "i"), name="mi")))
xg = T(np.arange(6.0).reshape(2, 3), ("g", "i")).bind(g="gpu")
print("all-gather :", [(c.kind, f"{c.bytes} B") for c in traffic(gather, {"x": xg}, mesh(2)).collectives])

distribute : ()
shard-view : ()
all-gather : [('all_gather', '24 B')]


## 7 · Per-device memory

The peak-memory simulator (notebook 10) gains one flag. With `local=True`, a
machine-bound dim counts as **one** — it indexes devices, not this device's
memory — so `peak_memory` reports the *per-device* footprint. The sharded block
peaks below its replicated self; that gap is precisely what tensor parallelism
buys.


In [10]:
full = peak_memory(placed.program, placed.inputs)
local = peak_memory(placed.program, placed.inputs, local=True)  # bound dims index devices
off = full.peak_bytes - local.peak_bytes
print(f"replicated peak : {full.peak_bytes} B")
print(f"per-device peak : {local.peak_bytes} B   ({off} B carried by the mesh, not this device)")

replicated peak : 4400 B
per-device peak : 2576 B   (1824 B carried by the mesh, not this device)


## 8 · The placed backward

Gradients carry their primal's placement — by the **same construction** that
carries charts. Every cotangent contribution is restamped as it is recorded, and
`bind` simply joined the restamp; backward `repeat`s that re-create a reduced mesh
dim re-declare its binding (or the backward would misalign against still-bound
forward operands). Two consequences, both checkable:

1. **the placed backward is the erased backward, bit-for-bit** — the erasure
   invariant extends through AD for free;
2. **gradients inherit placement**: $\partial L/\partial(\text{sharded})$ is
   sharded, $\partial L/\partial(\text{replicated})$ is replicated (no ZeRO-style
   resharding is modeled — CONCERNS #28).

We also spot-check against finite differences: the rebind is what keeps that from
misaligning.


In [11]:
def with_loss(m):
    # a scalar loss (sum of squares) tacked onto a zoo block's output
    return Program(m.program.instrs + (
        I("zsq", "pointwise", (m.out, m.out), f="mul"),
        I("zloss", "reduce", ("zsq",), f="sum", dims=("t", "d")),
    ))


jp_p, g_p = grad(with_loss(placed), "zloss", placed.inputs)
jp_e, g_e = grad(with_loss(erased), "zloss", erased.inputs)
ep, ee = run(jp_p, placed.inputs), run(jp_e, erased.inputs)
for v in ("x", "wq", "w2", "b1"):
    order = placed.inputs[v].names
    same = np.array_equal(ep[g_p[v]].to_numpy(order=order), ee[g_e[v]].to_numpy(order=order))
    print(f"grad {v:>3}: placed == erased bit-exact = {same}")

print("grad wq g.level :", ep[g_p["wq"]].layout.dim("g").level, " (sharded weight -> sharded grad)")
print("grad x  levels  :", [d.level for d in ep[g_p["x"]].layout.dims], " (replicated input -> replicated grad)")

fd = numeric_grad(with_loss(placed), "zloss", "x", placed.inputs)  # would misalign without the rebind
print("grad x vs finite-diff  max|Δ| =", float(np.max(np.abs(ep[g_p["x"]].to_numpy(order=("t", "d")) - fd))))

grad   x: placed == erased bit-exact = True


grad  wq: placed == erased bit-exact = True
grad  w2: placed == erased bit-exact = True
grad  b1: placed == erased bit-exact = True
grad wq g.level : gpu  (sharded weight -> sharded grad)
grad x  levels  : [None, None]  (replicated input -> replicated grad)


grad x vs finite-diff  max|Δ| = 1.1367009999929678e-08


### The training-step traffic — honest and unfused

Run the traffic pass over the *joint* forward+backward program and every backward
collective is just an **adjoint reduce**. The forward pair (the two output
projections) reappears, and each broadcast chain that introduced `g` in the
forward — q, k, v, and the MLP up-projection — contributes one input-gradient
all-reduce in the backward. That is **6** all-reduces.

The honest caveat (PLACEMENT.md; CONCERNS #28): the reference is **unfused**.
Megatron's paper fuses attention's three input-gradient reductions into one via
its conjugate *f*/*g* operators, giving 4. Collective fusion is a recorded *later
optimization* — a schedule over the same pieces — **not** a modeling error. The
reference layer's job is to be the honest ruler, not the fast path.


In [12]:
fwd = traffic(with_loss(placed), placed.inputs, mesh(2))
joint = traffic(jp_p, placed.inputs, mesh(2))
print("forward collectives :", len(fwd.collectives), " (Megatron's forward all-reduce pair)")
print("joint   collectives :", len(joint.collectives), "  all", {c.kind for c in joint.collectives})
print("   forward pair + one adjoint all-reduce per broadcast chain (q, k, v, mlp-up):")
for c in joint.collectives:
    print(f"     {c.var:>6}  {c.bytes} B")

forward collectives : 2  (Megatron's forward all-reduce pair)
joint   collectives : 6   all {'all_reduce'}
   forward pair + one adjoint all-reduce per broadcast chain (q, k, v, mlp-up):
          o  192 B
         m2  192 B
       %g58  192 B
      %g227  192 B
      %g246  192 B
      %g265  192 B


## 9 · The finale — data parallelism falls out

Nothing below is a new feature. It is the *same machinery*, pointed at the batch
dimension. Bind the batch dim `n` to the mesh and replicate the weight `w` by a
`repeat` over `n`. Then:

- **forward:** the loss reduces over the bound batch → an all-reduce (the usual
  loss aggregation).
- **backward:** the weight's gradient is the adjoint of that `repeat`, and
  $\text{repeat}^\dagger = \text{reduce-sum}$. The gradient sums over the bound
  batch dim — which **is** the data-parallel gradient all-reduce.

The weight-gradient all-reduce that every DP training loop performs was never
written. It *emerges* as the adjoint of a broadcast over a bound dim. And because
`w` was replicated, its gradient comes back replicated — ready for the next step.


In [13]:
dp = Program((
    I("x", "input"),
    I("w", "input"),
    I("wr", "repeat", ["w"], name="n", extent=(0, 4)),  # broadcast w across the batch mesh
    I("wb", "bind", ["wr"], levels={"n": "gpu"}),
    I("p", "pointwise", ["x", "wb"], f="mul"),
    I("zloss", "reduce", ["p"], f="sum", dims=("n", "i")),
))
inputs = {
    "x": T(np.arange(12.0).reshape(4, 3), ("n", "i")).bind(n="gpu"),  # sharded batch
    "w": T(np.array([1.0, 2.0, 3.0]), ("i",)),                        # replicated weight
}
jp, g = grad(dp, "zloss", inputs)
rep_dp = traffic(jp, inputs, mesh(4))
for c in rep_dp.collectives:
    print(f"  {c.var:>6}  {c.kind}  {c.bytes} B/device")
env = run(jp, inputs)
print("  grad w =", env[g["w"]].to_numpy(), " ==  x.sum(over batch) =", inputs["x"].to_numpy().sum(axis=0))
print("  grad w levels:", [d.level for d in env[g["w"]].layout.dims], "(replicated grad, ready for the next step)")

   zloss  all_reduce  12 B/device
    %g13  all_reduce  36 B/device
  grad w = [18. 22. 26.]  ==  x.sum(over batch) = [18. 22. 26.]
  grad w levels: [None] (replicated grad, ready for the next step)


## 10 · Loud refusals, scope, and the road to L4

The model refuses everything it does not cover, rather than guessing (D17). The
traffic pass raises on **lattice surgery over a bound dim** (slice/shift/select/…),
on a bound level the **machine doesn't have**, and on a **mesh extent exceeding
the level**. Each refusal is a place a silent wrong answer cannot exist.


In [14]:
xb = T(np.zeros((2, 3)), ("g", "i")).bind(g="gpu")
try:  # lattice surgery on a bound dim
    traffic(Program((I("x", "input"), I("s", "slice", ["x"], ranges={"g": (0, 1)}))), {"x": xb}, mesh(2))
except NotImplementedError as e:
    print("slice a bound dim  ->", e)
try:  # a bound level the machine lacks
    traffic(Program((I("x", "input"),)), {"x": xb}, Machine(()))
except KeyError as e:
    print("unknown level      ->", e)
try:  # a mesh extent bigger than the level
    traffic(Program((I("x", "input"),)), {"x": xb}, mesh(1))
except ValueError as e:
    print("extent > mesh      ->", e)

slice a bound dim  -> slice on machine-bound dim 'g' is not modeled (unbind first)
unknown level      -> "machine has no level 'gpu' (levels: [])"
extent > mesh      -> dim g: mesh extent 2 exceeds level 'gpu' count 1


### What L3-lite settled — and what L4 adds

The scope is written down plainly (CONCERNS #28): ring all-reduce / all-gather
formulas only; distribution and shard-views cost zero (weights assumed pre-placed,
loading unmodeled); no overlap or topology (α-β over a single per-level link);
gradients carry bindings but the backward is unfused (6 where fused f/g give 4);
no resharding / ZeRO, no per-device program extraction. Coarse, and *loudly* so —
every gap either refuses or is named in the ledger.

What made all of it possible was one field on a `Dim`. And that is the whole point
of the depth-ready schema (PLACEMENT.md D-C): **L4 is the same split+bind move at a
smaller depth.** Where L3 splits a semantic dim and binds a part to a *device*
mesh, L4 splits it again and binds a part to *SMEM* — tiling. Same mechanism,
different level of the tree; the flash-attention derivation is exactly this move
one tier down (LEVELS.md L4). The machine grows a level; the algebra does not
change. Collectives were alignment fixes; tiles will be, too.
